In [1]:
# General dependencies
import numpy as np
from matplotlib import pyplot as plt
from scipy import stats
import pickle as pkl
import pandas as pd
from strfpy.timeFreq import timefreq_raw
from soundsig.sound import spec_colormap

In [7]:
# Local folders

data_foler = '/Users/frederictheunissen/Google Drive/My Drive/PowerData/ECoG'
data_file = 'ECOG_data.npz'

In [8]:
# Read the data
file_data = np.load(f'{data_foler}/{data_file}', allow_pickle=True)
ECOG_data = file_data['arr_0'].item()


In [9]:
print(list(ECOG_data.keys()))



['e75_raw_lfp', 'e75_chunk_lfp', 'e86_raw_lfp', 'e86_chunk_lfp', 'sample_rate']


In [10]:
ECOG_data['sample_rate']

400

In [11]:
ECOG_data['e75_chunk_lfp'].shape

(10, 6593)

In [17]:
# Generate spectrograms for each channel:
fbanbs = [10, 25, 50]

# Spectrogram paramteters.
stim_params = {}

stim_params['nstd'] = 6
stim_params['high_freq'] = 100
stim_params['low_freq'] = 0
stim_params['log'] = 1
stim_params['stim_rate'] = 10  # Sample rate of spectrogram

# Colormap and other parametersfor plotting spectrograms
spec_colormap()   # defined in sound.py
cmap = plt.get_cmap('SpectroColorMap')
DBNOISE = 80
plotFlg = False

nreps = ECOG_data['e75_chunk_lfp'].shape[0]

for fb in fbanbs:
    lfpREPS = []
    stim_params['fband'] = fb
    for ch in range(nreps):
        lfp = ECOG_data['e75_chunk_lfp'][ch, :]
        tfrep = timefreq_raw(lfp, ECOG_data['sample_rate'], 'ft', params=stim_params)
        lfpREPS.append(tfrep['spec'])
    

        if plotFlg:
            plt.figure(figsize=(10, 4))
            f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, dpi=100, figsize = (8,4))
            minSpect = tfrep['spec'].max()-DBNOISE
            maxB = tfrep['spec'].max()
            
            ax1.imshow(tfrep['spec'], extent=[tfrep['t'][0], tfrep['t'][-1], tfrep['f'][0], tfrep['f'][-1]],
                aspect='auto', interpolation='nearest', origin='lower', cmap=cmap, vmin=minSpect, vmax=maxB)
            ax1.set_ylim(0, 100)
            ax1.set_ylabel('Frequency (Hz)')

            tval = np.arange(stop=len(lfp))/ECOG_data['sample_rate']
            ax2.plot(tval, lfp)
            ax2.set_xlabel('Time (s)')
            plt.show()
    outName = f'{data_foler}/ECOG_spec_fband{fb}.pkl'
    with open(outName, 'wb') as f:
        pkl.dump(lfpREPS, f)


/Users/frederictheunissen/opt/anaconda3/envs/strf/lib/python3.9/site-packages/soundsig/sound.py:541: UserWarning: Overwriting the cmap 'SpectroColorMap' that was already in the registry.
  mpl.colormaps.register(cmap=spec_cmap, force=True)
